# Data Ingestion Stage

In [17]:
from collections import namedtuple

from cnnClassifier.constants import CONFIG_FILE_PATH
from cnnClassifier.entity.config_entity import DataIngestionConfig

from src.CNNClassifier.constants import PARAM_FILE_PATH

In [18]:
import os

In [19]:
DataIngestionConfig = namedtuple('DataIngestionConfig',["root_dir","source_URL","local_data_file",
                                                        "unzip_dir"])

In [20]:
import os
import sys
from pathlib import Path

# Move from research/ to the project root, then add src/
project_root = Path(os.getcwd()).parent
sys.path.append(str(project_root / "src"))

In [21]:
from CNNClassifier.constants import *

In [22]:
from CNNClassifier.utils import read_yaml, create_directories

Import successful!


In [23]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.append(str(project_root / "src"))

In [25]:
from pathlib import Path

CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")

In [27]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config



In [28]:
import os
import urllib.request as request
from zipfile import ZipFile

In [29]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )


    def _get_updated_list_of_files(self, list_of_files):
        return [f for f in list_of_files if f.endswith(".jpg") and ("Cat" in f or "Dog" in f)]


    def _preprocess(self, zf: ZipFile, f: str, working_dir: str):
        target_filepath = os.path.join(working_dir, f)
        if not os.path.exists(target_filepath):
            zf.extract(f, working_dir)

        if os.path.getsize(target_filepath) == 0:
            os.remove(target_filepath)




    def unzip_and_clean(self):
        with ZipFile(file=self.config.local_data_file, mode="r") as zf:
            list_of_files = zf.namelist()
            updated_list_of_files = self._get_updated_list_of_files(list_of_files)
            for f in updated_list_of_files:
                self._preprocess(zf, f, self.config.unzip_dir)

In [31]:
from pathlib import Path

print(Path(CONFIG_FILE_PATH).exists())
print(Path(PARAMS_FILE_PATH).exists())

False
False


In [32]:
import os

print(os.getcwd())

D:\CNN\CNN\research


In [33]:
import os

os.chdir("D:/CNN/CNN")
print(os.getcwd())

D:\CNN\CNN


In [34]:
from pathlib import Path

CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")

In [37]:
from pathlib import Path
import yaml
from box import ConfigBox
from box.exceptions import BoxValueError

from CNNClassifier import logger


def read_yaml(path_to_yaml: Path):
    with open(path_to_yaml, "r") as yaml_file:
        content = yaml.safe_load(yaml_file)

        print("Content:", content)   # Debug

        return ConfigBox(content)

In [38]:
from pathlib import Path
import yaml
from box import ConfigBox
from box.exceptions import BoxValueError

from CNNClassifier import logger


def read_yaml(path_to_yaml: Path):
    try:
        with open(path_to_yaml, "r") as yaml_file:
            content = yaml.safe_load(yaml_file)

            if content is None:
                raise ValueError(f"{path_to_yaml} is empty.")

            logger.info(f"YAML file '{path_to_yaml}' loaded successfully")
            return ConfigBox(content)

    except BoxValueError:
        raise ValueError("YAML file is not in a valid format.")

    except Exception as e:
        raise e

In [39]:
from pathlib import Path
import yaml
from box import ConfigBox
from box.exceptions import BoxValueError

from CNNClassifier import logger


def read_yaml(path_to_yaml: Path):
    try:
        with open(path_to_yaml, "r") as yaml_file:
            content = yaml.safe_load(yaml_file)

            if content is None:
                raise ValueError(f"{path_to_yaml} is empty.")

            logger.info(f"YAML file '{path_to_yaml}' loaded successfully")
            return ConfigBox(content)

    except BoxValueError:
        raise ValueError("YAML file is not in a valid format.")

    except Exception as e:
        raise e

In [43]:
def create_directories(path_to_directories, verbose=True):
    for path in path_to_directories:
        os.makedirs(path, exist_ok=True)

        if verbose:
            logger.info(f"created directory at: {path}")

In [44]:
import os

print(os.path.exists("artifacts"))
print(os.path.exists("artifacts/data_ingestion"))

True
False


In [48]:
import os

print(os.path.exists("artifacts"))
print(os.path.exists("artifacts/data_ingestion"))

True
True


In [47]:
import os

os.makedirs("artifacts/data_ingestion", exist_ok=True)

In [49]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.unzip_and_clean()
except Exception as e:
    raise e

[2026-07-07 19:41:17,355:INFO:1250697027:YAML file 'config\config.yaml' loaded successfully]
[2026-07-07 19:41:17,357:INFO:1250697027:YAML file 'params.yaml' loaded successfully]
[2026-07-07 19:41:17,357:INFO:2694782583:created directory at: artifacts]
[2026-07-07 19:41:17,358:INFO:2694782583:created directory at: artifacts_root/data_ingestion]
